# Exercises in Fairness in Machine Learning

## Exercise 1

For this exercise, we will use the `adult` dataset (available on moodle or from the [UCI Machine Learning repository](https://archive.ics.uci.edu/dataset/2/adult)). Do the following:

1. Load in the dataset and correct the error in the income column (replace the "." with the empty string such that there are only two categories).
2. Create an X dataset using the variables "age", "workclass", "education", "occupation", "race", "sex", "hours-per-week". For the categorical variables with missing values, replace the missing values with a new category "Unknown". Also replace any values that are "?" with the value "Unknown" (using `str.replace`, for instance)
3. Turn the five categorical variables in X into dummy variables and remove the original five variables (This will probably give you around 44 columns in X)
4. Create the y variable by using the following code `y = pd.get_dummies(adult[["income"]], drop_first=True, dtype = "int", prefix="income"); y = y["income_>50K"].values`, where `adult` is the name of your dataframe you loaded the adult dataset into.
5. Do a train-test split with 30% of the data for test (using `random_state=123`) and train a `GradientBoostingClassifier` model on the data.
6. Evaluate your model using various evaluation metrics and look at the confusion matrix of your model
7. To be able to calculate the various fairness metrics in regard to the variable `sex`, we need to construct two separate confusion matrices for the test dataset, one for `female` and one for `male`. First, create separate test sets for `female` and `male` as well as the predicted values for each gender. That is, create `X_test_female`, `X_test_male`, `y_test_female`, `y_test_male`, `y_pred_female`, and `y_pred_male`. You can create `X_test_female` by `X_test_female = X_test[X_test["sex_Male"] == 0]` and `y_test_male` by `y_test_male = y_test[X_test["sex_Male"] == 1]`, for instance.
8. We can now create the True Positive (TP), True Negative (TN), False Positive (FP), and False Negative (FN) for each gender. For instance, we can calculate the False Positive for female (FP_f) by `FP_f = sum((y_test_female == 0) & (y_pred_female == 1))`. Calculate the eight values `TP_f`, `TN_f`, `FP_f`, `FN_f`, `TP_m`, `TN_m`, `FP_m`, and `FN_m`.
9. Calculate the accuracy for female and male and comment on the results
10. Is there error rate balance, i.e. are the false positive rate (FPR) and false negative rate (FNR) the same across the two genders?
11. Is there predictive parity?
12. Is there Statistical parity?
15. [Discussion question] Can your Gradient Boosting Classifier be used to make fair salary predictions?
13. [Discussion question] In what sense is the `adult` dataset biased (unfair)?
14. [Discussion question] If the dataset is biased, where could the bias potentially come from?

In [8]:
import pandas as pd
# Load the adult dataset
adult_data = pd.read_csv('adult.csv')

# Check the first few rows of the dataset
adult_data.head()
adult_data['income'] = adult_data['income'].str.replace(' ', '')


In [9]:
# Replace missing values in categorical columns with 'Unknown'
categorical_columns = ['workclass', 'education', 'occupation', 'race', 'sex']
adult_data[categorical_columns] = adult_data[categorical_columns].apply(lambda col: col.replace('?', 'Unknown'))

# Create the X dataset with the required columns
X = adult_data[['age', 'workclass', 'education', 'occupation', 'race', 'sex', 'hours-per-week']]

# Replace missing values with 'Unknown' for categorical variables
X[categorical_columns] = X[categorical_columns].fillna('Unknown')

# Turn the five categorical variables into dummy variables
X = pd.get_dummies(X, drop_first=True)
X.head()

,age,hours-per-week,workclass_Local-gov,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,workclass_State-gov,workclass_Unknown,workclass_Without-pay,...,occupation_Protective-serv,occupation_Sales,occupation_Tech-support,occupation_Transport-moving,occupation_Unknown,race_Asian-Pac-Islander,race_Black,race_Other,race_White,sex_Male
0,39,40,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,True,True
1,50,13,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,True,True
2,38,40,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,True
3,53,40,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,True
4,28,40,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False


In [10]:
# Create the y variable
y = pd.get_dummies(adult_data[['income']], drop_first=True, dtype=int)
y = y['income_>50K'].values

In [11]:
from sklearn.model_selection import train_test_split

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)

In [12]:
from sklearn.ensemble import GradientBoostingClassifier

# Initialize the model
gb_model = GradientBoostingClassifier(random_state=123)

# Train the model
gb_model.fit(X_train, y_train)

,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are'friedman_mse' for the mean squared error with improvement score byFriedman, 'squared_error' for mean squared error. The default value of'friedman_mse' is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, 

In [19]:
# Predictive Parity check: Comparing False Positive Rates and False Negative Rates for both genders

print("FPR for Female:", FPR_female)
print("FNR for Female:", FNR_female)

print("FPR for Male:", FPR_male)
print("FNR for Male:", FNR_male)

FPR for Female: 0.0006625441696113075
FNR for Female: 0.9877300613496932
FPR for Male: 0.03375366195389123
FNR for Male: 0.8413757700205339


In [20]:
# Statistical Parity check: Comparing the proportion of positive predictions for both genders

# Proportion of predicted >50K for Female
positive_female = sum(y_pred_female == 1) / len(y_pred_female)

# Proportion of predicted >50K for Male
positive_male = sum(y_pred_male == 1) / len(y_pred_male)

print("Proportion of predicted >50K for Female:", positive_female)
print("Proportion of predicted >50K for Male:", positive_male)

Proportion of predicted >50K for Female: 0.001442109600329625
Proportion of predicted >50K for Male: 0.058577405857740586


In [13]:
from sklearn.metrics import accuracy_score, confusion_matrix

# Make predictions on the test set
y_pred = gb_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

# Confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("Confusion Matrix:")
print(conf_matrix)

Accuracy: 0.8478809800040947
Confusion Matrix:
[[12111   268]
 [ 1961   313]]


In [14]:
# Create confusion matrices for male and female
X_test_female = X_test[X_test['sex_Male'] == 0]
y_test_female = y_test[X_test['sex_Male'] == 0]
y_pred_female = gb_model.predict(X_test_female)

X_test_male = X_test[X_test['sex_Male'] == 1]
y_test_male = y_test[X_test['sex_Male'] == 1]
y_pred_male = gb_model.predict(X_test_male)

# Confusion matrix for females
conf_matrix_female = confusion_matrix(y_test_female, y_pred_female)

# Confusion matrix for males
conf_matrix_male = confusion_matrix(y_test_male, y_pred_male)

print("Confusion Matrix for Female:")
print(conf_matrix_female)

print("Confusion Matrix for Male:")
print(conf_matrix_male)

Confusion Matrix for Female:
[[4525    3]
 [ 322    4]]
Confusion Matrix for Male:
[[7586  265]
 [1639  309]]


In [15]:
# For Female
TP_f = sum((y_test_female == 1) & (y_pred_female == 1))
TN_f = sum((y_test_female == 0) & (y_pred_female == 0))
FP_f = sum((y_test_female == 0) & (y_pred_female == 1))
FN_f = sum((y_test_female == 1) & (y_pred_female == 0))

# For Male
TP_m = sum((y_test_male == 1) & (y_pred_male == 1))
TN_m = sum((y_test_male == 0) & (y_pred_male == 0))
FP_m = sum((y_test_male == 0) & (y_pred_male == 1))
FN_m = sum((y_test_male == 1) & (y_pred_male == 0))

# Print values
print("For Female:")
print(f"TP_f: {TP_f}, TN_f: {TN_f}, FP_f: {FP_f}, FN_f: {FN_f}")

print("For Male:")
print(f"TP_m: {TP_m}, TN_m: {TN_m}, FP_m: {FP_m}, FN_m: {FN_m}")

For Female:
TP_f: 4, TN_f: 4525, FP_f: 3, FN_f: 322
For Male:
TP_m: 309, TN_m: 7586, FP_m: 265, FN_m: 1639


In [16]:
accuracy_female = (TP_f + TN_f) / (TP_f + TN_f + FP_f + FN_f)
accuracy_male = (TP_m + TN_m) / (TP_m + TN_m + FP_m + FN_m)

print("Accuracy for Female:", accuracy_female)
print("Accuracy for Male:", accuracy_male)

Accuracy for Female: 0.9330449114132674
Accuracy for Male: 0.8056944586182263


In [17]:
FPR_female = FP_f / (FP_f + TN_f)
FNR_female = FN_f / (FN_f + TP_f)

FPR_male = FP_m / (FP_m + TN_m)
FNR_male = FN_m / (FN_m + TP_m)

print("FPR for Female:", FPR_female)
print("FNR for Female:", FNR_female)

print("FPR for Male:", FPR_male)
print("FNR for Male:", FNR_male)

FPR for Female: 0.0006625441696113075
FNR for Female: 0.9877300613496932
FPR for Male: 0.03375366195389123
FNR for Male: 0.8413757700205339


In [18]:
# Calculate False Positive Rate (FPR) and False Negative Rate (FNR) for Female and Male

# For Female
FPR_female = FP_f / (FP_f + TN_f)
FNR_female = FN_f / (FN_f + TP_f)

# For Male
FPR_male = FP_m / (FP_m + TN_m)
FNR_male = FN_m / (FN_m + TP_m)

print("FPR for Female:", FPR_female)
print("FNR for Female:", FNR_female)

print("FPR for Male:", FPR_male)
print("FNR for Male:", FNR_male)

FPR for Female: 0.0006625441696113075
FNR for Female: 0.9877300613496932
FPR for Male: 0.03375366195389123
FNR for Male: 0.8413757700205339


In [ ]:
13.Can your Gradient Boosting Classifier be used to make fair salary predictions?

No, the Gradient Boosting Classifier cannot necessarily be used to make fair salary predictions without addressing potential biases in the data. The model learns from historical data that may reflect societal biases, such as gender, race, or other factors influencing salaries unfairly. If these biases are present in the dataset, the model could reinforce them, leading to unfair predictions. To ensure fairness in salary predictions, additional fairness metrics and debiasing techniques should be incorporated into the model training process.

In [ ]:
14.In what sense is the adult dataset biased (unfair)?

The adult dataset is biased in several ways, most notably in its representation of demographic factors like gender, race, and work class. For instance, there is often a disproportionate number of males compared to females in high-paying job categories, which could result in a model that predicts higher salaries for men compared to women. Additionally, certain racial and ethnic groups may be underrepresented or face systemic barriers that could impact their earnings. These imbalances make the dataset unfair and reflective of societal biases.

In [ ]:
15.If the dataset is biased, where could the bias potentially come from?

The bias in the dataset could originate from several sources. First, historical societal inequalities, such as wage gaps based on gender and race, could have shaped the data in ways that reflect past discrimination. Second, sampling bias might occur if the dataset does not adequately represent all demographic groups, leading to overrepresentation of certain groups (e.g., white, male workers) and underrepresentation of others (e.g., women, minorities). Finally, there could be biases in the data collection process itself, where certain types of jobs or industries that are more likely to pay higher salaries might not capture the diversity of workers accurately. These biases could then be perpetuated when used to train predictive models.